# mu-logsigma-encoder-head — ex1: single-head Linear + chunk into mu and logsigma

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `mu-logsigma-encoder-head`. Running the final beacon cell reports progress against the `VAE: mu+logsigma encoder head` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: mu+logsigma encoder head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mu-logsigma-encoder-head`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mu-logsigma-encoder-head"
DD_SUBTOPIC = "VAE: mu+logsigma encoder head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## mu + logsigma encoder head — quick refresher

A VAE encoder ends in TWO heads — one for `mu`, one for `logsigma` — both of shape `(B, latent_dim)`. The cheapest implementation is a SINGLE `nn.Linear(D, 2 * latent_dim)` followed by `x.chunk(2, dim=-1)`:

```python
params = self.head(features)            # (B, 2 * latent_dim)
mu, logsigma = params.chunk(2, dim=-1)  # each (B, latent_dim)
```

**Why `logsigma`, not `sigma`.** A neural net output is unconstrained — it can be negative. Predicting `logsigma` and exponentiating later (`sigma = logsigma.exp()`) guarantees `sigma > 0` without any clamping. Predicting `sigma` directly would need a softplus or an abs, which behave badly near zero.

**Compared to two separate `Linear` heads.** Functionally identical (both are `(D -> latent_dim)` affine maps). The single-head + chunk form saves one parameter object and reads more clearly as `'encoder emits the parameters of a Gaussian'`. ARENA uses this form.

### Exercise 1 — single-head Linear + chunk into mu and logsigma

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply a single `nn.Linear(D, 2*latent_dim)` followed by `chunk(2, dim=-1)` to project encoder features into the Gaussian parameters `(mu, logsigma)`, each of shape `(B, latent_dim)`.
> Keywords: vae, encoder, chunk, gaussian-head
> ```

**KCs targeted:** `encoder-double-width-linear`, `chunk-split-mu-logsigma`

Implement `ex1_encoder_head(features, weight, bias, latent_dim)`. The double-width-Linear-plus-chunk pattern every VAE encoder uses:

1. `features` has shape `(B, D)` — output of the encoder body, just before the Gaussian-parameter heads.
2. `weight` has shape `(2 * latent_dim, D)` and `bias` has shape `(2 * latent_dim,)` — together they parameterize an `nn.Linear(D, 2 * latent_dim)`.
3. Compute the affine: `params = features @ weight.T + bias` → shape `(B, 2 * latent_dim)`.
4. Split with `params.chunk(2, dim=-1)` into `mu` and `logsigma`, each `(B, latent_dim)`.
5. Return the tuple `(mu, logsigma)`.

Input: `features` `(B, D)`, `weight` `(2*latent_dim, D)`, `bias` `(2*latent_dim,)`, `latent_dim` `int`.
Output: tuple `((B, latent_dim), (B, latent_dim))`.

The visualization runs your head on random features for `latent_dim=8` and renders the per-batch mu and logsigma as two side-by-side heatmaps — useful for spotting dead latent dims at a glance.

In [ ]:
def ex1_encoder_head(features: Tensor, weight: Tensor, bias: Tensor, latent_dim: int) -> tuple[Tensor, Tensor]:
    params = features @ weight.T + bias   # (B, 2 * latent_dim)
    mu, logsigma = params.chunk(2, dim=-1)
    return mu, logsigma


<details><summary>Solution</summary>

```python
def ex1_encoder_head(features: Tensor, weight: Tensor, bias: Tensor, latent_dim: int) -> tuple[Tensor, Tensor]:
    params = features @ weight.T + bias   # (B, 2 * latent_dim)
    mu, logsigma = params.chunk(2, dim=-1)
    return mu, logsigma
```

**`chunk(2, dim=-1)` vs slicing.** `params.chunk(2, dim=-1)` is equivalent to `(params[..., :latent_dim], params[..., latent_dim:])` — but it doesn't hard-code the split point, and reads as 'split in two along the last axis'. If you ever change `latent_dim`, the slicing breaks; the chunk doesn't.

**Why one Linear, not two.** Functionally identical, but the single-head form is one parameter object and one matmul instead of two — easier to checkpoint, faster on hardware. ARENA's reference VAE uses this convention.

**The heatmap is the diagnostic.** A dead latent dim shows up as a vertical stripe of constant logsigma near 0 (the encoder has given up on that dim). A healthy VAE shows varied magnitudes across dims and samples.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()